# Workforce Attrition Patterns & Risk Hotspot Analysis
**Organization:** Palo Alto Networks

This notebook follows the Unified Mentor analytical methodology:
1. Data Validation & Cleaning
2. Overall Attrition Assessment
3. Department & Role-Wise Analysis
4. Demographic Attrition Analysis
5. Tenure & Career Stage Analysis
6. Workload & Mobility Impact Analysis

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'dashboard'))

from utils import (
    attrition_rate,
    department_role_heatmap_data,
    early_tenure_attrition,
    grouped_attrition_rate,
    load_cleaned_data,
    workload_attrition_index,
)

CHARTS_DIR = PROJECT_ROOT / 'outputs' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

df = load_cleaned_data(force_refresh=True)
print('Dataset shape:', df.shape)
print('Overall attrition rate:', round(attrition_rate(df), 2), '%')

## Step 1 — Data Validation & Cleaning

In [ ]:
print('Missing values in key columns:')
print(df.isna().sum().sort_values(ascending=False).head(10))
print('\nAttrition label distribution:')
print(df['Attrition'].value_counts())
print('\nDepartments:', df['Department'].unique())
print('Job roles:', df['JobRole'].nunique())

## Step 2 — Overall Attrition Assessment

In [ ]:
overall_rate = attrition_rate(df)
retained = (df['Attrition'] == 0).sum()
exited = (df['Attrition'] == 1).sum()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Retained', 'Exited'], [retained, exited], color=['#2ECC71', '#E74C3C'])
ax.set_title(f'Overall Attrition: {overall_rate:.2f}%')
ax.set_ylabel('Employee Count')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'overall_attrition.png', dpi=150)
plt.show()

## Step 3 — Department & Role-Wise Analysis

In [ ]:
dept = grouped_attrition_rate(df, 'Department')
role = grouped_attrition_rate(df, 'JobRole')

print('Department attrition rates:')
display(dept)
print('\nTop roles by attrition:')
display(role.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=dept, x='Department', y='Attrition Rate (%)', ax=ax, palette='Reds_r')
ax.set_title('Attrition Rate by Department')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'department_attrition.png', dpi=150)
plt.show()

heatmap = department_role_heatmap_data(df)
plt.figure(figsize=(12, 4))
sns.heatmap(heatmap, annot=True, fmt='.1f', cmap='Reds')
plt.title('Department x Role Attrition Heatmap (%)')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'department_role_heatmap.png', dpi=150)
plt.show()

## Step 4 — Demographic Attrition Analysis

In [ ]:
for col, filename in [
    ('AgeGroup', 'age_group_attrition.png'),
    ('Gender', 'gender_attrition.png'),
    ('MaritalStatus', 'marital_status_attrition.png'),
    ('EducationField', 'education_field_attrition.png'),
]:
    summary = grouped_attrition_rate(df, col)
    print(f'\n{col}:')
    display(summary)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=summary, x=col, y='Attrition Rate (%)', ax=ax)
    ax.set_title(f'Attrition by {col}')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / filename, dpi=150)
    plt.show()

## Step 5 — Tenure & Career Stage Analysis

In [ ]:
tenure = grouped_attrition_rate(df, 'TenureBand')
promotion = grouped_attrition_rate(df, 'PromotionStagnation')
early = early_tenure_attrition(df, years=2)

print(f'Early-tenure attrition (<=2 years): {early:.2f}%')
display(tenure)
display(promotion)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=tenure, x='TenureBand', y='Attrition Rate (%)', ax=axes[0])
axes[0].set_title('Attrition by Tenure Band')
sns.barplot(data=promotion, x='PromotionStagnation', y='Attrition Rate (%)', ax=axes[1])
axes[1].set_title('Promotion Stagnation Impact')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'tenure_promotion_attrition.png', dpi=150)
plt.show()

## Step 6 — Workload & Mobility Impact Analysis

In [ ]:
workload = workload_attrition_index(df)
distance = grouped_attrition_rate(df, 'DistanceBand')

display(workload)
display(distance)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, category in zip(axes, ['Overtime', 'Business Travel']):
    subset = workload[workload['Category'] == category]
    sns.barplot(data=subset, x='Factor', y='Attrition Rate (%)', ax=ax)
    ax.set_title(f'Attrition by {category}')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'workload_attrition.png', dpi=150)
plt.show()

print('\nAll charts saved to:', CHARTS_DIR)

## Step 7 — Predictive Attrition Modeling

In [ ]:
from modeling import train_attrition_model

results = train_attrition_model(df)
print('Model performance:')
for metric, value in results.metrics.items():
    print(f'  {metric}: {value:.4f}')

print('\nClassification report:')
print(results.classification_report)

display(results.feature_importance.head(15))

In [ ]:
plt.figure(figsize=(8, 6))
top = results.feature_importance.head(15)
sns.barplot(data=top, y='Feature', x='Importance', palette='Reds_r')
plt.title('Top 15 Attrition Predictors')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'feature_importance.png', dpi=150)
plt.show()